<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 4.1: FIRRTL 介绍

**上一节: [生成器: 类型](3.6_types.ipynb)**<br>
**下一节: [FIRRTL AST 遍历](4.2_firrtl_ast_traversal.ipynb)**

## 动机
你已经学习了一些 Scala 并编写了一些 Chisel，对于 90% 的用户来说，这足以成为 Chisel 爱好者。

然而，某些用例更适合表达为 Chisel 设计的程序化转换，而不是作为生成器。

例如，假设我们想要计算设计中的寄存器数量。这很难作为生成器来完成，因此我们可以编写一个 FIRRTL 传递来为我们完成这个任务。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.iotesters.{ChiselFlatSpec, Driver, PeekPokeTester}
import firrtl._

## 什么是 FIRRTL？
你可能已经意识到，当你执行一个 Chisel 设计时，它会详细化（执行周围的 Scala 代码）以构建你的生成器实例，并解析所有 Scala 参数。

Chisel 不是直接生成 Verilog，而是生成一个称为 FIRRTL 的中间表示，它代表了详细化（参数解析后）的 RTL 实例。它可以被序列化（转换为字符串以写入文件），并且这种序列化语法是人类可读的。然而在内部，它不是表示为长字符串。相反，它是一个组织为节点树的数据结构，称为抽象语法树（AST）。

让我们来看看！我们将采用一个简单的 Chisel 设计，详细化它，并检查它生成什么 FIRRTL！

首先，我们定义一个 Chisel 模块，它将输入信号延迟两个周期。

In [ ]:
class DelayBy2(width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  val r0 = RegNext(io.in)
  val r1 = RegNext(r0)
  io.out := r1
}

接下来，让我们详细化它，序列化并打印出它生成的 FIRRTL。

In [ ]:
println(chisel3.Driver.emit(() => new DelayBy2(4)))

如你所见，序列化的 FIRRTL 看起来非常像我们的 Chisel 设计，所有生成器参数都已解析。

## FIRRTL AST

如前所述，FIRRTL 表示可以序列化为字符串，但在内部，它是一个称为 AST（抽象语法树）的数据结构。这个数据结构是一个节点树，其中一个节点可以包含子节点。这个数据结构中没有循环。

让我们看看内部数据结构是什么样子的：

In [ ]:
val firrtlSerialization = chisel3.Driver.emit(() => new DelayBy2(4))
val firrtlAST = firrtl.Parser.parse(firrtlSerialization.split("\n").toIterator, Parser.GenInfo("file.fir"))

println(firrtlAST)

Obviously, the serialization of a datastructure isn't as pretty, but you can see some of the classes and such that internally represent the RTL 设计. Let's try to pretty that up a bit to make it understandable.

In [ ]:
println(stringifyAST(firrtlAST))

这是保存 FIRRTL AST 的内部数据结构。它是一个树结构，其根节点是 **电路**，它有 3 个子节点：**@[file.fir@2.0]**、**ArrayBuffer** 和 **cmd5WrapperHelperDelayBy2**。以下是序列化的 `电路` 的实际 Scala 类定义：<a name="电路"></a><img src="images/电路.png" alt="电路 case 类" />



如您所见，它有三个子节点：`info: Info`、`Modules: Seq[DefModule]` 和 `main: String`。它扩展了 `FirrtlNode`，所有 FIRRTL AST 节点都必须这样做。暂时忽略 `def mapXXXX` 函数。

许多 FIRRTL 节点包含一个 `info: Info` 字段，解析器可以插入文件信息（如行号和列号），或插入 `NoInfo` 标记。在这个例子中，**@[file.fir@2.0]** 指的是 FIRRTL 文件第 2 行第 0 列。

以下部分将详细概述所有这些 FIRRTL 节点。

# FIRRTL 节点描述

本节描述了在 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) 中找到的常见 FirrtlNodes。

有关此处未提及的组件的更多详细信息，请参阅 [FIRRTL 规范](https://github.com/ucb-bar/firrtl/blob/master/spec/spec.pdf)。


## 电路
电路是任何 Firrtl 数据结构的根节点。始终只有一个电路，该电路包含模块定义列表和顶层模块的名称。

#### FirrtlNode Declaration
```scala 
电路(info: Info, modules: Seq[DefModule], main: String)
```

#### Concrete Syntax
```
电路 Adder:
  ... //List of modules
```
#### In-memory Representation
```scala
电路(NoInfo, Seq(...), "Adder")
```

## 模块

模块是 Firrtl 中的模块化单元，永远不会直接嵌套（声明模块的实例有其自己的具体语法和 AST 表示）。每个模块都有一个名称、一个端口列表和一个包含其实现的主体。

#### FirrtlNode declaration
```scala
模块(info: Info, name: String, ports: Seq[Port], body: Stmt) extends DefModule
```

#### Concrete Syntax
```
模块 Adder:
  ... // list of ports
  ... // statements
```
#### In-memory representation
```scala
模块(NoInfo, "Adder", Seq(...), )
```

## Port
端口定义了模块IO的一部分，具有名称、方向（输入或输出）和类型。

#### FirrtlNode Declaration
```scala
类 Port(info: Info, name: String, direction: Direction, tpe: 类型)
```
#### Concrete Syntax
```
输入 x: UInt
```

#### In-memory representation
```scala
Port(NoInfo, "x", 输入, UIntType(UnknownWidth))
```

## Statement
语句用于描述模块内的组件以及它们如何交互。以下是一些常用的语句：

### Block of Statements
一组语句。通常用作模块声明中的body字段。

### 导线 Declaration
导线声明包含名称和类型。它既可以是源（连接自）也可以是汇（连接至）。
#### FirrtlNode declaration
```scala
DefWire(info: Info, name: String, tpe: 类型)
```
#### Concrete syntax
```
导线 w: UInt
```
#### In-memory Representation
```scala
DefWire(NoInfo, "w", UIntType(UnknownWidth))
```

### 寄存器 Declaration
A 寄存器 declaration, containing a name, 类型, 时钟 signal, 复位 signal, and 复位 值.
#### FirrtlNode declaration
```scala
DefRegister(info: Info, name: String, tpe: 类型, 时钟: Expression, 复位: Expression, init: Expression)
```

### Connection
表示从源到汇的有向连接。请注意，它遵循最后连接语义，如 Chisel 中所述。

#### FirrtlNode declaration
```scala
Connect(info: Info, loc: Expression, expr: Expression)
```

### Other Statements
其他语句类型如 `DefMemory`、`DefNode`、`IsInvalid`、`Conditionally` 等在此省略；更多详细信息请参阅 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/freechipsproject/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala)。

## Expression
表达式表示对已声明组件的引用或逻辑和算术运算。以下是一些常用的表达式：

### Reference
对已声明组件（如导线、寄存器或端口）的引用。它具有名称和类型字段。请注意，它不包含指向实际声明的指针，而是仅包含作为字符串的名称。

#### FirrtlNode declaration
```scala
Reference(name: String, tpe: 类型)
```

### DoPrim
匿名原始操作，例如 `Add`、`Sub`、`And`、`Or` 或子字选择 (`Bits`)。操作类型由 `op: PrimOp` 字段指示。请注意，所需参数和常量的数量由 `op` 决定。

#### FirrtlNode declaration
```scala
DoPrim(op: PrimOp, args: Seq[Expression], consts: Seq[BigInt], tpe: 类型)
```

### Other Expressions
其他表达式包括 `SubField`、`SubIndex`、`SubAccess`、`Mux`、`ValidIf` 等，在 [firrtl/src/main/scala/firrtl/ir/IR.scala](https://github.com/ucb-bar/firrtl/blob/master/src/main/scala/firrtl/ir/IR.scala) 和 [FIRRTL 规范](https://github.com/ucb-bar/firrtl/blob/master/spec/spec.pdf) 中有更详细的描述。

# Back to our 示例

让我们再次查看示例中的 FIRRTL AST。希望设计的结构现在更加清晰！

In [ ]:
println(stringifyAST(firrtlAST))

本节到此结束！在下一节中，我们将了解 FIRRTL 转换如何遍历此 AST 并修改它。